# Prompt 22: User Defined Functions (UDFs)
## Databricks Certified Associate Developer for Apache Spark
### Topic 3 — DataFrame and Column Operations (30%)

---

## What is a UDF?

A **User Defined Function (UDF)** lets you apply arbitrary Python logic to DataFrame columns. Spark can't optimise the function itself — it treats the UDF as a black box — so UDFs carry a performance cost compared to built-in functions.

**Preference hierarchy (always prefer left to right):**
```
Built-in (pyspark.sql.functions) >> Pandas UDF (@pandas_udf) >> Regular UDF (udf())
```

---

## Creating a Regular UDF

```python
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# Lambda form
my_udf = udf(lambda x: x.upper() if x else None, StringType())

# Named function form
def to_upper(s):
    return s.upper() if s else None

my_udf = udf(to_upper, StringType())

# Decorator form
@udf(returnType=StringType())
def to_upper(s):
    return s.upper() if s else None
```

**Critical rule:** You MUST specify the `returnType`. Spark needs it to build the query plan. If omitted it defaults to `StringType()` — which will silently return wrong results for numeric UDFs.

---

## Applying a UDF

```python
# .withColumn
df.withColumn('name_upper', my_udf(col('name')))

# .select
df.select('id', my_udf(col('name')).alias('name_upper'))

# .filter — works but is especially slow (row-by-row with Python)
df.filter(my_udf(col('name')) == 'ALICE')
```

---

## Registering a UDF for SQL

```python
# Register by name for use in spark.sql() and selectExpr()
spark.udf.register('to_upper_sql', to_upper, StringType())

# Use in SQL string
spark.sql("SELECT id, to_upper_sql(name) AS name_upper FROM people").show()

# Use in selectExpr
df.selectExpr("id", "to_upper_sql(name) AS name_upper").show()
```

---

## Why UDFs Are Slow

| Factor | Built-in function | Regular UDF | Pandas UDF |
|--------|------------------|-------------|------------|
| Catalyst optimisation | ✅ full | ❌ black box | ❌ black box |
| Execution model | JVM vectorised | Python row-by-row | Python vectorised (Arrow) |
| Serialisation | None | Pickle each row (JVM↔Python) | Apache Arrow batch |
| Typical overhead | Baseline | 2–10× slower | ~2× slower than built-in |

---

## Pandas UDFs (Vectorised UDFs)

```python
from pyspark.sql.functions import pandas_udf
import pandas as pd

# SCALAR Pandas UDF: Series → Series
@pandas_udf(StringType())
def to_upper_pd(s: pd.Series) -> pd.Series:
    return s.str.upper()

df.withColumn('name_upper', to_upper_pd(col('name'))).show()
```

Key points:
- Input is a `pandas.Series`, output is a `pandas.Series`
- Type hints are **required** (or pass `returnType` to decorator)
- Uses Apache Arrow for zero-copy serialisation between JVM and Python
- Much faster than regular UDFs for CPU-bound operations
- Requires `pyarrow` installed

---

## NULL Handling Inside UDFs

```python
# WRONG — will throw NullPointerException when Spark passes None
@udf(StringType())
def bad_udf(s):
    return s.upper()   # AttributeError: 'NoneType' has no attribute 'upper'

# CORRECT — always guard against None
@udf(StringType())
def good_udf(s):
    if s is None:
        return None
    return s.upper()

# Pythonic short form
@udf(StringType())
def good_udf(s):
    return s.upper() if s is not None else None
```

Spark maps Python `None` return values to SQL `NULL`.

In [ ]:
# Cell 1: Setup and simple regular UDF
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf, pandas_udf, lit
from pyspark.sql.types import (
    StringType, IntegerType, DoubleType, BooleanType,
    ArrayType, StructType, StructField
)
import pyspark.sql.functions as F
import pandas as pd
import re

spark = SparkSession.builder \
    .appName('UDFGuide') \
    .master('local[2]') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

# Sample data
people = [
    (1, 'alice smith',   'alice@example.com',  28, 72000.0),
    (2, 'BOB JONES',     'bob@company.org',    35, 95000.0),
    (3, 'Carol  White',  None,                 None, 61000.0),
    (4, 'dave brown',    'dave@example.com',   42, None),
    (5, None,            'unknown@x.com',      55, 110000.0),
]
schema = StructType([
    StructField('id',     IntegerType(), False),
    StructField('name',   StringType(),  True),
    StructField('email',  StringType(),  True),
    StructField('age',    IntegerType(), True),
    StructField('salary', DoubleType(),  True),
])
df = spark.createDataFrame(people, schema=schema)
df.show()

# ---------------------------------------------------------------
# Simple UDF — title-case a name, guard against None
# ---------------------------------------------------------------
def title_case(s):
    if s is None:
        return None
    return ' '.join(w.capitalize() for w in s.strip().split())

title_udf = udf(title_case, StringType())

print('=== Apply UDF with withColumn ===')
df.withColumn('name_clean', title_udf(col('name'))).show()

# ---------------------------------------------------------------
# Decorator form — same function
# ---------------------------------------------------------------
@udf(returnType=StringType())
def extract_domain(email):
    if email is None:
        return None
    parts = email.split('@')
    return parts[1] if len(parts) == 2 else None

print('=== Decorator-form UDF — extract email domain ===')
df.withColumn('domain', extract_domain(col('email'))).select('id', 'email', 'domain').show()

# ---------------------------------------------------------------
# Lambda UDF
# ---------------------------------------------------------------
salary_band = udf(lambda s: 'high' if s and s > 80000 else ('low' if s else None), StringType())

print('=== Lambda UDF — salary band ===')
df.withColumn('band', salary_band(col('salary'))).select('id', 'salary', 'band').show()

In [ ]:
# Cell 2: Registering UDFs for SQL and selectExpr

# ---------------------------------------------------------------
# Register Python function for SQL usage
# ---------------------------------------------------------------
def title_case_sql(s):
    if s is None:
        return None
    return ' '.join(w.capitalize() for w in s.strip().split())

spark.udf.register('title_case', title_case_sql, StringType())
spark.udf.register('extract_domain', lambda e: e.split('@')[1] if e and '@' in e else None, StringType())

df.createOrReplaceTempView('people')

# ---------------------------------------------------------------
# Use registered UDF in spark.sql()
# ---------------------------------------------------------------
print('=== spark.sql() using registered UDF ===')
spark.sql("""
    SELECT
        id,
        title_case(name)     AS name_clean,
        extract_domain(email) AS email_domain
    FROM people
    WHERE name IS NOT NULL
""").show()

# ---------------------------------------------------------------
# Use registered UDF in selectExpr()
# ---------------------------------------------------------------
print('=== selectExpr() using registered UDF ===')
df.selectExpr(
    'id',
    'title_case(name) AS name_clean',
    'extract_domain(email) AS domain',
    'salary'
).show()

# ---------------------------------------------------------------
# UDF registered via the DataFrame API is also available in SQL
# (decorator-form UDF can also be registered)
# ---------------------------------------------------------------
spark.udf.register('extract_domain2', extract_domain)  # reuse the decorator UDF
spark.sql("SELECT id, extract_domain2(email) AS domain FROM people").show()

In [ ]:
# Cell 3: Complex return types — ArrayType and StructType from a UDF

# ---------------------------------------------------------------
# UDF returning ArrayType — split name into parts
# ---------------------------------------------------------------
split_name_udf = udf(
    lambda s: s.strip().split() if s else [],
    ArrayType(StringType())
)

print('=== UDF returning ArrayType ===')
df_parts = df.withColumn('name_parts', split_name_udf(col('name')))
df_parts.select('id', 'name', 'name_parts').show(truncate=False)

# Access array elements from the UDF result
df_parts.select(
    'id',
    col('name_parts')[0].alias('first_name'),
    col('name_parts')[1].alias('last_name')
).show()

# ---------------------------------------------------------------
# UDF returning StructType — parse email into components
# ---------------------------------------------------------------
email_struct_type = StructType([
    StructField('user',   StringType(), True),
    StructField('domain', StringType(), True),
    StructField('tld',    StringType(), True),
])

from pyspark.sql import Row

def parse_email(email):
    if email is None or '@' not in email:
        return None
    user, rest = email.split('@', 1)
    parts = rest.rsplit('.', 1)
    domain = parts[0] if len(parts) == 2 else rest
    tld    = parts[1] if len(parts) == 2 else None
    return Row(user=user, domain=domain, tld=tld)

parse_email_udf = udf(parse_email, email_struct_type)

print('=== UDF returning StructType ===')
df_email = df.withColumn('email_parsed', parse_email_udf(col('email')))
df_email.select('id', 'email', 'email_parsed').show(truncate=False)
df_email.printSchema()

# Access nested struct fields from UDF result
df_email.select(
    'id',
    col('email_parsed.user').alias('email_user'),
    col('email_parsed.domain').alias('email_domain'),
    col('email_parsed.tld').alias('email_tld'),
).show()

In [ ]:
# Cell 4: Pandas UDF (Vectorised UDF) — faster via Apache Arrow

# ---------------------------------------------------------------
# SCALAR Pandas UDF: pandas.Series → pandas.Series
# ---------------------------------------------------------------

@pandas_udf(StringType())
def title_case_pd(s: pd.Series) -> pd.Series:
    return s.apply(lambda x: ' '.join(w.capitalize() for w in x.strip().split()) if pd.notna(x) else None)

print('=== Pandas UDF — title_case_pd ===')
df.withColumn('name_clean', title_case_pd(col('name'))).show()

# ---------------------------------------------------------------
# Pandas UDF with numeric column
# ---------------------------------------------------------------
@pandas_udf(DoubleType())
def apply_tax(salary: pd.Series) -> pd.Series:
    return salary * 0.75   # 25% tax — vectorised, no loop needed

print('=== Pandas UDF — apply_tax (numeric) ===')
df.withColumn('net_salary', apply_tax(col('salary'))).select('id', 'salary', 'net_salary').show()

# ---------------------------------------------------------------
# Pandas UDF with multiple input columns
# ---------------------------------------------------------------
@pandas_udf(StringType())
def build_label(name: pd.Series, age: pd.Series) -> pd.Series:
    return name.fillna('Unknown') + ' (age: ' + age.fillna(0).astype(int).astype(str) + ')'

print('=== Pandas UDF — multiple input columns ===')
df.withColumn('label', build_label(col('name'), col('age'))).select('id', 'name', 'age', 'label').show()

# ---------------------------------------------------------------
# Why Pandas UDF is faster: Arrow eliminates row-by-row pickling
# ---------------------------------------------------------------
print('Preference hierarchy summary:')
print('  1st: pyspark.sql.functions built-ins      — JVM, Catalyst optimised, fastest')
print('  2nd: @pandas_udf                           — Arrow batch, ~2× overhead vs built-in')
print('  3rd: udf()                                 — row-by-row Python pickle, 2-10× overhead')
print()
print('Use UDF when NO built-in covers the logic.')
print('Use Pandas UDF for numeric/vectorisable logic where built-ins are insufficient.')

In [ ]:
# Cell 5: NULL handling, performance demo, and UDF pitfalls

# ---------------------------------------------------------------
# Null handling — demonstrate the difference
# ---------------------------------------------------------------

# BAD: will raise AttributeError on None
@udf(StringType())
def unsafe_upper(s):
    return s.upper()   # NullPointerException if s is None

# GOOD: guard against None
@udf(StringType())
def safe_upper(s):
    return s.upper() if s is not None else None

print('=== Safe UDF (None guard) ===')
# Only run safe version; unsafe_upper would fail on row 5 (name=None)
df.withColumn('name_upper', safe_upper(col('name'))).select('id', 'name', 'name_upper').show()

# ---------------------------------------------------------------
# Compare: built-in vs UDF for the same operation
# ---------------------------------------------------------------
print('=== Built-in F.upper() — ALWAYS prefer this over UDF for simple transforms ===')
df.withColumn('name_upper_builtin', F.upper(col('name'))).select('id', 'name', 'name_upper_builtin').show()
# F.upper() is JVM-native — no Python overhead, no serialisation

# ---------------------------------------------------------------
# Demonstrate: UDF return type MUST match actual return value
# ---------------------------------------------------------------
print('=== Return type mismatch causes silent NULL ===')
# This UDF returns an int but is declared as StringType — result is NULL
wrong_type_udf = udf(lambda x: 42, StringType())   # 42 → cast to string '42' actually works
# More dangerous: declaring DoubleType but returning a string
bad_return_udf = udf(lambda x: 'not_a_number', DoubleType())  # returns NULL silently
df.withColumn('bad', bad_return_udf(col('id'))).select('id', 'bad').show()

# ---------------------------------------------------------------
# Registering a Pandas UDF for SQL (also possible)
# ---------------------------------------------------------------
spark.udf.register('apply_tax_sql', apply_tax)
spark.sql("SELECT id, salary, apply_tax_sql(salary) AS net_salary FROM people").show()

spark.stop()
print('Done.')

## Real-World Scenarios

<details>
<summary>Scenario 1: Normalising free-text product names using a UDF when no built-in covers the logic</summary>

**Situation:**
A retail pipeline ingests product names from a CSV file. Names are inconsistently cased (`'APPLE WATCH'`, `'apple watch'`, `'Apple  Watch'`). The requirement is to produce a canonical `slug` column: lowercase, single spaces, replace spaces with hyphens. No single built-in handles all three steps.

**Solution:**
```python
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType
import re

@udf(StringType())
def slugify(name):
    if name is None:
        return None
    return re.sub(r'\s+', '-', name.strip().lower())

df.withColumn('slug', slugify(col('product_name'))).show()
# 'APPLE  WATCH' → 'apple-watch'
# None           → None  (safe)
```

**Exam Sub-topic:** Defining a UDF; null safety; when built-ins aren't sufficient
</details>

<details>
<summary>Scenario 2: Registering a UDF for SQL-style reporting in a Databricks notebook</summary>

**Situation:**
A data analyst writes Spark SQL cells in a Databricks notebook. They need a custom function `risk_band(score)` that maps a numeric credit score (0–850) to `'low'`, `'medium'`, or `'high'`. The analyst wants to use it in `WHERE` clauses and `SELECT` lists via SQL syntax.

**Solution:**
```python
from pyspark.sql.types import StringType

def risk_band(score):
    if score is None:
        return None
    if score < 580:
        return 'high'
    elif score < 720:
        return 'medium'
    return 'low'

spark.udf.register('risk_band', risk_band, StringType())

customers.createOrReplaceTempView('customers')

# Now use in SQL
spark.sql("""
    SELECT customer_id,
           credit_score,
           risk_band(credit_score) AS risk
    FROM customers
    WHERE risk_band(credit_score) = 'high'
    ORDER BY credit_score
""").show()
```

**Exam Sub-topic:** `spark.udf.register()`; using UDFs in `spark.sql()` and `selectExpr()`
</details>

<details>
<summary>Scenario 3: UDF returning a StructType to parse a complex string column</summary>

**Situation:**
A log DataFrame has a `raw_event` column containing strings like `'LOGIN|user123|2024-01-15T08:30:00'`. You need to parse this into separate typed fields: `event_type` (string), `user_id` (string), `event_ts` (string). No single built-in handles pipe-delimited parsing into a struct.

**Solution:**
```python
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql import Row
from pyspark.sql.functions import udf, col

event_schema = StructType([
    StructField('event_type', StringType(), True),
    StructField('user_id',    StringType(), True),
    StructField('event_ts',   StringType(), True),
])

@udf(event_schema)
def parse_event(raw):
    if raw is None:
        return None
    parts = raw.split('|')
    if len(parts) != 3:
        return Row(event_type=None, user_id=None, event_ts=None)
    return Row(event_type=parts[0], user_id=parts[1], event_ts=parts[2])

df_parsed = logs.withColumn('event', parse_event(col('raw_event')))
df_parsed.select(
    'id',
    col('event.event_type'),
    col('event.user_id'),
    col('event.event_ts')
).show()
```

**Exam Sub-topic:** UDF with `StructType` return; accessing nested fields from UDF output
</details>

<details>
<summary>Scenario 4: Replacing a slow regular UDF with a Pandas UDF for a high-volume scoring pipeline</summary>

**Situation:**
A machine learning scoring pipeline applies a linear formula to 50 million rows. It was originally written as a regular UDF:
```python
score_udf = udf(lambda f1, f2, f3: 0.3*f1 + 0.5*f2 + 0.2*f3, DoubleType())
```
Stage runtime: 4 minutes. Goal: reduce to under 30 seconds by converting to a Pandas UDF.

**Solution:**
```python
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import DoubleType
import pandas as pd

# Note: first check if a built-in expression works here — it does!
# Prefer this over any UDF:
df.withColumn('score', col('f1')*0.3 + col('f2')*0.5 + col('f3')*0.2)

# If more complex (not expressible as built-in), use Pandas UDF:
@pandas_udf(DoubleType())
def score_pd(f1: pd.Series, f2: pd.Series, f3: pd.Series) -> pd.Series:
    return 0.3*f1 + 0.5*f2 + 0.2*f3

df.withColumn('score', score_pd(col('f1'), col('f2'), col('f3'))).show()
# ~8× faster than row-by-row UDF via Apache Arrow vectorisation
```

**Exam Sub-topic:** Pandas UDF (SCALAR) for vectorised operations; preference hierarchy (built-in first)
</details>

<details>
<summary>Scenario 5: UDF returning None silently vs NullPointerException — debugging null issues</summary>

**Situation:**
A pipeline adds a `category` column using a UDF. In development (small dataset, no NULLs) it works fine. In production (real data with NULL values), the job fails with a Python exception. The team needs to understand null handling in UDFs and fix the issue.

**Root cause and fix:**
```python
# BROKEN — crashes when name is NULL
@udf(StringType())
def categorise(name):
    return 'electronics' if 'phone' in name.lower() else 'other'
    # AttributeError: 'NoneType' object has no attribute 'lower'

# FIXED — always check for None first
@udf(StringType())
def categorise(name):
    if name is None:
        return None          # Spark maps Python None → SQL NULL
    return 'electronics' if 'phone' in name.lower() else 'other'

# Alternative: use F.when() + F.lower() built-ins — no UDF needed, faster:
df.withColumn(
    'category',
    F.when(F.lower(col('name')).contains('phone'), 'electronics').otherwise('other')
)
```

**Exam Sub-topic:** Null safety in UDFs; Python `None` → SQL `NULL`; prefer built-ins to avoid UDF pitfalls entirely
</details>

## Exam Cheat Sheet

| Question | Answer |
|----------|--------|
| Create a UDF from a function | `udf(my_func, ReturnType())` |
| Create a UDF from a lambda | `udf(lambda x: x.upper(), StringType())` |
| Decorator UDF syntax | `@udf(returnType=StringType())` |
| Apply UDF to a column | `df.withColumn('c', my_udf(col('x')))` |
| Register UDF for SQL | `spark.udf.register('name', fn, ReturnType())` |
| Use registered UDF in SQL | `spark.sql("SELECT my_udf(col) FROM t")` |
| Use registered UDF in selectExpr | `df.selectExpr("my_udf(col) AS c")` |
| Default return type if omitted | `StringType()` — silently wrong for numeric UDFs |
| UDF with ArrayType return | `udf(fn, ArrayType(StringType()))` |
| UDF with StructType return | `udf(fn, StructType([StructField(...)]))` |
| Why UDFs are slow | Row-by-row Python pickle serialisation; no Catalyst optimisation |
| Pandas UDF decorator | `@pandas_udf(ReturnType())` |
| Pandas UDF signature | `fn(s: pd.Series) -> pd.Series` (type hints required) |
| Why Pandas UDF is faster | Apache Arrow batch transfer — no per-row pickling |
| Null handling in UDFs | Always check `if s is None: return None` |
| Python None return maps to | SQL `NULL` |
| Preference hierarchy | Built-in > Pandas UDF > Regular UDF |
| Can Pandas UDF be registered for SQL? | Yes — `spark.udf.register('name', pandas_udf_fn)` |
| Wrong return type from UDF | Returns `NULL` silently (no error) |

---

## Practice Q&A

<details>
<summary>Q1: You write a UDF that returns an integer but forget to specify the return type. What happens?</summary>

**A:** If `returnType` is omitted, Spark defaults to `StringType()`. The UDF will run but will convert the integer return value to a string, or in some cases produce `NULL`. The column type will be `StringType` rather than `IntegerType`. **Always explicitly declare the return type.**

```python
# WRONG — defaults to StringType; numeric operations on result will fail
bad = udf(lambda x: len(x) if x else 0)

# CORRECT
good = udf(lambda x: len(x) if x else 0, IntegerType())
```
</details>

<details>
<summary>Q2: What is the difference between a regular UDF and a Pandas UDF?</summary>

**A:**
| | Regular UDF | Pandas UDF |
|--|-------------|------------|
| Input | One Python scalar per row | `pandas.Series` (batch) |
| Serialisation | Pickle each row (JVM ↔ Python) | Apache Arrow (zero-copy batch) |
| Speed | 2–10× slower than built-ins | ~2× slower than built-ins |
| Decorator | `@udf(ReturnType())` | `@pandas_udf(ReturnType())` |
| Type hint | Not required | Required (`Series → Series`) |

Choose Pandas UDF over regular UDF when the logic is vectorisable (math, string vectorised ops, etc.).
</details>

<details>
<summary>Q3: How do you use a UDF registered with spark.udf.register() inside selectExpr()?</summary>

**A:** After registering, call the UDF by its registered name as a SQL expression string:

```python
spark.udf.register('double_it', lambda x: x * 2, IntegerType())

# selectExpr — SQL string syntax
df.selectExpr('id', 'double_it(value) AS value_doubled').show()

# spark.sql — same registered name
spark.sql('SELECT id, double_it(value) AS value_doubled FROM my_table').show()
```
</details>

<details>
<summary>Q4: A UDF processes a column that contains NULL values and the job crashes. What is the fix?</summary>

**A:** Spark passes SQL `NULL` values to UDFs as Python `None`. The fix is to guard against `None` at the start of every UDF:

```python
@udf(StringType())
def safe_fn(s):
    if s is None:          # Always check first
        return None        # Return None → Spark writes NULL
    return s.strip().upper()
```

Returning Python `None` from a UDF maps to a SQL `NULL` value in the output column.
</details>

<details>
<summary>Q5: In what order should you prefer transformation options, and why?</summary>

**A:** The preference hierarchy is:

1. **Built-in `pyspark.sql.functions`** — JVM-native, Catalyst-optimised, fastest. Always check here first.
2. **Pandas UDF (`@pandas_udf`)** — Python but vectorised via Apache Arrow; ~2× slower than built-ins; good for complex vectorisable logic.
3. **Regular UDF (`udf()`)** — row-by-row Python; slowest option; use only when neither built-ins nor Pandas UDFs can express the logic.

Example: converting a name to uppercase:
- Built-in: `F.upper(col('name'))` ← **use this**
- UDF: `udf(lambda x: x.upper() if x else None, StringType())(col('name'))` ← unnecessary overhead
</details>